# matmul：从程序变换到 CPU 编译产物

对应 kickoff R02。使用固定源码、真实 CPU 和 NumPy float64 对照。当前环境带 `VERSION-SKEW`，native 产物不能归属于固定 XLA C++ revision。这个 Notebook 是新内核中的交互验证；原始 capture 的完整生产命令和字节指纹另存于 manifest。

先运行仓库根目录的 `python3 -B tools/sync-environment.py check`。原始样本由 `research/jax-stack/matmul_probe.py` 生成，以下单元读取 `cpu-matmul-003`，并重新执行核心数学运算。

In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "upstream-sources.lock").exists())
HERE = ROOT / "research/jax-stack"
CAPTURE = ROOT / "artifacts/jax-stack/cpu-matmul-003"
os.environ["JAX_PLATFORMS"] = "cpu"
verification = subprocess.run([sys.executable, "-B", str(HERE / "verify_research.py")], cwd=ROOT, check=True, text=True, capture_output=True)
print(verification.stdout.strip())
results = json.loads((HERE / "cpu-results.json").read_text())
index = json.loads((HERE / "source-index.json").read_text())

{"capture_id": "cpu-matmul-003", "source_entries_checked": 37, "source_edges_checked": 13, "artifact_count": 1062, "qualifiers": ["VERSION-SKEW"]}


## 1. 固定输入与数学参考

`A[4,8]`、`W[8,6]`、`X[3,4,8]`。batch 共享 W。损失是矩阵乘结果的平方和；对 A/X 和 W 都求梯度。

单样本：`dA = 2(AW)W.T`，`dW = 2A.T(AW)`。batch 的 dW 还需沿 batch 维度求和。

In [2]:
import jax
import jax.numpy as jnp
import numpy as np

with np.load(CAPTURE / "inputs.npz", allow_pickle=False) as archive:
    A, W, X = (archive[k] for k in ("a", "w", "x"))

def matmul(a, w):
    return jnp.matmul(a, w, precision=jax.lax.Precision.HIGHEST)

batched = jax.vmap(matmul, in_axes=(0, None))
def loss(a, w):
    return jnp.sum(matmul(a, w) ** 2)
def batch_loss(x, w):
    return jnp.sum(batched(x, w) ** 2)

a64, w64, x64 = (v.astype(np.float64) for v in (A, W, X))
y64, z64 = a64 @ w64, x64 @ w64
cases = [
    ("matmul", matmul, (A, W), y64),
    ("vmap", batched, (X, W), z64),
    ("grad", jax.grad(loss, argnums=(0, 1)), (A, W), (2*y64@w64.T, 2*a64.T@y64)),
    ("jit-grad-vmap", jax.grad(batch_loss, argnums=(0, 1)), (X, W), (2*z64@w64.T, 2*np.einsum("bmk,bmn->kn", x64, z64))),
]
for name, function, args, expected in cases:
    actual = jax.jit(function)(*args)
    jax.block_until_ready(actual)
    errors = []
    for observed, reference in zip(jax.tree.leaves(actual), jax.tree.leaves(expected), strict=True):
        np.testing.assert_allclose(np.asarray(observed), reference, rtol=2e-5, atol=2e-5)
        errors.append(float(np.max(np.abs(np.asarray(observed) - reference))))
    print(name, "PASS", "maximum absolute errors:", errors)
print("Environment:", jax.__version__, jax.devices())

matmul PASS maximum absolute errors: [5.102698930059546e-08]
vmap PASS maximum absolute errors: [8.485674118929865e-08]
grad PASS maximum absolute errors: [7.892122444452809e-08, 9.822404523074368e-08]
jit-grad-vmap PASS maximum absolute errors: [1.4270093773305348e-07, 1.1280561940107958e-07]
Environment: 0.11.2.dev20260830+5832e86644 [CpuDevice(id=0)]


## 2. Jaxpr、StableHLO 与导出 HLO

下面直接调用当前 JAX 的观察接口。导出 HLO 与 backend 原始 dump 是不同观测点。共同 W 的 vmap 样本可以用一个高秩收缩表示；不能把 vmap 的 batch 数当成实际发射次数。

In [3]:
print("Jaxpr for vmap:")
print(jax.make_jaxpr(batched)(X, W))
lowered = jax.jit(batched).lower(X, W)
print("StableHLO:")
print(lowered.as_text("stablehlo"))
print("Exported HLO:")
print(lowered.as_text("hlo"))

Jaxpr for vmap:
{ lambda ; a:f32[3,4,8] b:f32[8,6]. let
    c:f32[3,4,6] = dot_general[
      dimension_numbers=(([2], [0]), ([], []))
      precision=(Precision.HIGHEST, Precision.HIGHEST)
      preferred_element_type=float32
    ] a b
  in (c,) }
StableHLO:
module @jit_matmul attributes {mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<3x4x8xf32>, %arg1: tensor<8x6xf32>) -> (tensor<3x4x6xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg0, %arg1, contracting_dims = [2] x [0], precision = [HIGHEST, HIGHEST] : (tensor<3x4x8xf32>, tensor<8x6xf32>) -> tensor<3x4x6xf32>
    return %0 : tensor<3x4x6xf32>
  }
}

Exported HLO:
HloModule jit_matmul, entry_computation_layout={(f32[3,4,8]{2,1,0}, f32[8,6]{1,0})->f32[3,4,6]{2,1,0}}

ENTRY main.1 {
  a.1 = f32[3,4,8]{2,1,0} parameter(0)
  w.1 = f32[8,6]{1,0} parameter(1)
  ROOT dot_general.1 = f32[3,4,6]{2,1,0} dot(a.1, w.1), lhs_contracting_dims={2}, rhs_contracting_dims={

## 3. 实际 HLO passes 与代码生成

完整 capture 使用 `--xla_dump_hlo_pass_re=.+`。字面量 `.*` 在已核对源码中会跳过未改变 HLO 的 pass；`--xla_dump_emitter_re=mlir-fusion|llvm` 另行控制 emitter 中间产物。下面的模块编号来自本次文件与 header 匹配，不把它写成跨运行常量。

In [4]:
for case in results["cases"]:
    print(case["name"], "->", case["native_module_prefix"], "pass-boundary files:", case["native_pass_boundary_count"])
    for name in case["native_pass_boundaries"][:3]:
        print(" ", name)
print("Code generation artifacts:")
for path in sorted((CAPTURE / "xla-dump").glob("*.ll")):
    print(path.name)
print("Native objects:", [item["elf"] for item in results["objects"]])

matmul -> module_0004.jit_matmul pass-boundary files: 146
  module_0004.jit_matmul.0000.async-collective.after_pipeline-start.before_async-collective-custom-call-rewriter.txt
  module_0004.jit_matmul.0001.async-collective.after_async-collective-custom-call-rewriter.before_async-collective-replacer.txt
  module_0004.jit_matmul.0002.async-collective.after_async-collective-replacer.before_pipeline-end.txt
vmap_matmul -> module_0014.jit_matmul pass-boundary files: 162
  module_0014.jit_matmul.0000.async-collective.after_pipeline-start.before_async-collective-custom-call-rewriter.txt
  module_0014.jit_matmul.0001.async-collective.after_async-collective-custom-call-rewriter.before_async-collective-replacer.txt
  module_0014.jit_matmul.0002.async-collective.after_async-collective-replacer.before_pipeline-end.txt
grad_matmul -> module_0024.jit_loss pass-boundary files: 162
  module_0024.jit_loss.0000.async-collective.after_pipeline-start.before_async-collective-custom-call-rewriter.txt
  modul

## 4. 从产物回到源码

索引记录固定 revision、输入输出、约束、调用方和被调用方。native 源码条目仍标为 SOURCE-ONLY；相同函数名或 pass 名不能消除 VERSION-SKEW。

In [5]:
selected = {"jax.matmul", "jax.dot-lowering", "jax.backend-compile", "jaxlib.compile", "xla.pass-pipeline", "xla.cpu-buffers"}
for entry in index["entries"]:
    if entry["id"] in selected:
        print(entry["id"], entry["path"], "line", entry["line"])
        print(" inputs:", entry["inputs"])
        print(" outputs:", entry["outputs"])
        print(" constraints:", entry["constraint"])
        print(" callers/callees:", entry["callers"], entry["callees"])

jax.matmul upstream/jax/jax/_src/numpy/tensor_contractions.py line 138
 inputs: 数组 lhs/rhs、precision、preferred_element_type、out_sharding
 outputs: 按广播与收缩维度定义的数组
 constraints: 检查秩、batch 维度与 sharding；构造 dimension_numbers 后调用 lax.dot_general。
 callers/callees: [] ['jax.dot-general']
jax.dot-lowering upstream/jax/jax/_src/lax/lax.py line 6264
 inputs: lowering context、MLIR lhs/rhs、dimension_numbers 和 precision
 outputs: stablehlo.dot_general 及必要转换
 constraints: 构造 DotDimensionNumbers 和 precision_config；输出仍是 MLIR value。
 callers/callees: [] []
jax.backend-compile upstream/jax/jax/_src/compiler.py line 335
 inputs: backend Client、MLIR module、设备、CompileOptions、host callbacks
 outputs: LoadedExecutable 或编译异常
 constraints: 真实 backend 与 CompileOnlyPyClient 分支不同；普通路径调用 backend.compile_and_load。
 callers/callees: [] ['jaxlib.compile']
jaxlib.compile upstream/jax/jaxlib/py_client.cc line 475
 inputs: MLIR module、设备和编译参数
 outputs: PyLoadedExecutable
 constraints: 克隆 module，包装 HloProgram/IFRT 编译选项；这是

## 5. cost、buffer assignment 与测量边界

静态 FLOPs/bytes 不是实际 DRAM 流量；compiler temporary storage 不是设备峰值。目标 roofline、split 降峰值和 overlap 仍需对应的硬件输入、运行测量与真实业务实验。

In [6]:
for case in results["cases"]:
    costs = case["cost_after_optimization"]
    memory = case["memory_analysis"]
    print(case["name"], {"flops": costs.get("flops"), "estimated_bytes_accessed": costs.get("bytes accessed"), "compiler_temp_bytes": memory.get("temp_size_in_bytes")})
print("Tracing:", json.loads((CAPTURE / "tracing.json").read_text())["observations"])
print("Separate native compilation completion events:", results["cache_compilation_completions"])

matmul {'flops': 384.0, 'estimated_bytes_accessed': 416.0, 'compiler_temp_bytes': 0}
vmap_matmul {'flops': 1152.0, 'estimated_bytes_accessed': 864.0, 'compiler_temp_bytes': 0}
grad_matmul {'flops': 1176.0, 'estimated_bytes_accessed': 1456.0, 'compiler_temp_bytes': 96}
jit_grad_vmap_matmul {'flops': 3600.0, 'estimated_bytes_accessed': 3760.0, 'compiler_temp_bytes': 608}
Tracing: [{'call': 1, 'lhs_shape': [4, 8], 'trace_count': 1}, {'call': 2, 'lhs_shape': [4, 8], 'trace_count': 1}, {'call': 3, 'lhs_shape': [5, 8], 'trace_count': 2}]
Separate native compilation completion events: 2


## 未完成的研究

完整清单见 `PLAN.md`。本 Notebook 没有完成匹配源码构建、Pallas TPU 编译、实际推理图 fusion/Pallas 注入、split 降低实测峰值、通信 overlap 或 XProf 自定义事件验证。模型、源码修改边界与 TPU 条件仍待确认。